# Sales Forecasting Dashboard — Rossmann Stores

## What we build:
An interactive dashboard with 4 sections:
1. KPI Overview — total sales, avg sales, best store
2. Sales Trends — monthly and weekly patterns
3. Store Performance — top vs bottom stores
4. Forecast View — model predictions vs actual

## Library: Plotly Dash
- Interactive charts (zoom, filter, hover)
- Runs as local web app
- Professional looking UI

## Step 1 — Import Libraries

In [1]:
# ── Dashboard libraries ───────────────────────────────────────────
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go

# ── Data libraries ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

✅ Libraries loaded!


## Step 2 — Load Data & Model
Loading cleaned data and trained LightGBM model
for dashboard visualizations and predictions.

In [2]:
# ── Load cleaned data ─────────────────────────────────────────────
df = pd.read_csv('../data/processed/train_cleaned.csv',
                 low_memory=False)
df['Date'] = pd.to_datetime(df['Date'])

# ── Load feature engineered data ─────────────────────────────────
df_features = pd.read_csv('../data/processed/train_features.csv',
                           low_memory=False)

# ── Load trained model ────────────────────────────────────────────
with open('../src/model.pkl', 'rb') as f:
    model = pickle.load(f)

# ── Filter open stores only ───────────────────────────────────────
df_open = df[df['Open'] == 1].copy()

# ── Add time columns ──────────────────────────────────────────────
df_open['Year']  = df_open['Date'].dt.year
df_open['Month'] = df_open['Date'].dt.month
df_open['Week']  = df_open['Date'].dt.isocalendar().week.astype(int)

# ── Verify ────────────────────────────────────────────────────────
print(f"✅ Cleaned data  : {df_open.shape}")
print(f"✅ Feature data  : {df_features.shape}")
print(f"✅ Model loaded  : LightGBM {model.best_iteration} iterations")
print()
print("✅ All data and model loaded successfully!")

✅ Cleaned data  : (844392, 21)
✅ Feature data  : (813172, 35)
✅ Model loaded  : LightGBM 672 iterations

✅ All data and model loaded successfully!


## Step 3 — Prepare Dashboard Data
Pre-computing all aggregations needed for dashboard.
Pre-computing is faster than computing inside dashboard callbacks.

In [3]:
# ── KPI Data ──────────────────────────────────────────────────────
total_sales    = df_open['Sales'].sum()
avg_daily_sales= df_open['Sales'].mean()
total_customers= df_open['Customers'].sum()
total_stores   = df_open['Store'].nunique()
best_store     = df_open.groupby('Store')['Sales'].sum().idxmax()
best_store_sales = df_open.groupby('Store')['Sales'].sum().max()

print("📊 KPI Summary:")
print(f"   Total Sales     : €{total_sales/1e6:.1f}M")
print(f"   Avg Daily Sales : €{avg_daily_sales:,.0f}")
print(f"   Total Customers : {total_customers/1e6:.1f}M")
print(f"   Total Stores    : {total_stores}")
print(f"   Best Store      : Store {best_store} (€{best_store_sales/1e6:.1f}M)")

# ── Monthly Sales Trend ───────────────────────────────────────────
monthly_sales = df_open.groupby(
    ['Year', 'Month'])['Sales'].sum().reset_index()
monthly_sales['Date'] = pd.to_datetime(
    monthly_sales['Year'].astype(str) + '-' +
    monthly_sales['Month'].astype(str))
monthly_sales = monthly_sales.sort_values('Date')

# ── Store Performance ─────────────────────────────────────────────
store_perf = df_open.groupby('Store').agg(
    Total_Sales    = ('Sales', 'sum'),
    Avg_Sales      = ('Sales', 'mean'),
    Total_Customers= ('Customers', 'sum')
).reset_index().sort_values('Total_Sales', ascending=False)

# ── Promo Impact ──────────────────────────────────────────────────
promo_sales = df_open.groupby('Promo')['Sales'].mean().reset_index()
promo_sales['Promo'] = promo_sales['Promo'].map(
    {0: 'No Promo', 1: 'Promo'})

# ── Day of week sales ─────────────────────────────────────────────
dow_sales = df_open.groupby('DayOfWeek')['Sales'].mean().reset_index()
dow_sales['DayName'] = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

# ── Store type sales ──────────────────────────────────────────────
storetype_sales = df_open.groupby('StoreType')['Sales'].mean().reset_index()

print()
print("✅ All dashboard data prepared!")

📊 KPI Summary:
   Total Sales     : €5873.2M
   Avg Daily Sales : €6,956
   Total Customers : 644.0M
   Total Stores    : 1115
   Best Store      : Store 262 (€19.5M)

✅ All dashboard data prepared!


## Step 4 — Build Dashboard
Building interactive Plotly Dash dashboard with 4 sections.

In [4]:
# ── Initialize app ────────────────────────────────────────────────
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# ── Color scheme ──────────────────────────────────────────────────
COLORS = {
    'primary'   : '#2196F3',
    'success'   : '#4CAF50',
    'warning'   : '#FF9800',
    'danger'    : '#F44336',
    'dark'      : '#263238',
    'light'     : '#F5F5F5'
}

# ── KPI Card component ────────────────────────────────────────────
def kpi_card(title, value, subtitle, color):
    return dbc.Card([
        dbc.CardBody([
            html.H6(title, style={'color': '#666', 'fontSize': '13px'}),
            html.H3(value, style={'color': color, 'fontWeight': 'bold'}),
            html.P(subtitle, style={'color': '#999', 'fontSize': '12px', 'margin': '0'})
        ])
    ], style={'borderLeft': f'4px solid {color}', 'borderRadius': '8px'})

# ── Layout ────────────────────────────────────────────────────────
app.layout = dbc.Container([

    # ── Header ───────────────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            html.H2("Rossmann Sales Forecasting Dashboard",
                    style={'color': COLORS['dark'], 'fontWeight': 'bold'}),
            html.P("Interactive analytics and ML-powered sales forecasting",
                   style={'color': '#666'})
        ])
    ], style={'padding': '20px 0 10px 0'}),

    html.Hr(),

    # ── KPI Row ───────────────────────────────────────────────────
    dbc.Row([
        dbc.Col(kpi_card("Total Sales", f"€{total_sales/1e6:.0f}M",
                         "Jan 2013 — Jul 2015", COLORS['primary']), width=3),
        dbc.Col(kpi_card("Avg Daily Sales", f"€{avg_daily_sales:,.0f}",
                         "Per store per day", COLORS['success']), width=3),
        dbc.Col(kpi_card("Total Customers", f"{total_customers/1e6:.0f}M",
                         "Total visits", COLORS['warning']), width=3),
        dbc.Col(kpi_card("Best Store", f"Store {best_store}",
                         f"€{best_store_sales/1e6:.1f}M total", COLORS['danger']), width=3),
    ], style={'marginBottom': '20px'}),

    # ── Sales Trend & Day of Week ─────────────────────────────────
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("Monthly Sales Trend"),
                dbc.CardBody([
                    dcc.Graph(id='monthly-trend',
                        figure=px.line(
                            monthly_sales, x='Date', y='Sales',
                            color='Year' if 'Year' in monthly_sales else None,
                            title='',
                            labels={'Sales': 'Total Sales (€)'}
                        ).update_layout(
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            margin=dict(t=10, b=10)
                        )
                    )
                ])
            ])
        ], width=8),
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("Avg Sales by Day"),
                dbc.CardBody([
                    dcc.Graph(id='dow-chart',
                        figure=px.bar(
                            dow_sales, x='DayName', y='Sales',
                            color='Sales',
                            color_continuous_scale='Blues',
                            labels={'Sales': 'Avg Sales (€)'}
                        ).update_layout(
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            margin=dict(t=10, b=10),
                            showlegend=False
                        )
                    )
                ])
            ])
        ], width=4),
    ], style={'marginBottom': '20px'}),

    # ── Store Performance & Promo Impact ─────────────────────────
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("Top 15 Stores by Total Sales"),
                dbc.CardBody([
                    dcc.Graph(id='store-perf',
                        figure=px.bar(
                            store_perf.head(15),
                            x='Store', y='Total_Sales',
                            color='Total_Sales',
                            color_continuous_scale='Greens',
                            labels={'Total_Sales': 'Total Sales (€)'}
                        ).update_layout(
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            margin=dict(t=10, b=10)
                        )
                    )
                ])
            ])
        ], width=8),
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("Promo vs No Promo"),
                dbc.CardBody([
                    dcc.Graph(id='promo-chart',
                        figure=px.bar(
                            promo_sales, x='Promo', y='Sales',
                            color='Promo',
                            color_discrete_map={
                                'No Promo': COLORS['danger'],
                                'Promo'   : COLORS['success']},
                            labels={'Sales': 'Avg Sales (€)'}
                        ).update_layout(
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            margin=dict(t=10, b=10),
                            showlegend=False
                        )
                    )
                ])
            ])
        ], width=4),
    ], style={'marginBottom': '20px'}),

    # ── Forecast Section ──────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("Sales Forecast — Select Store"),
                dbc.CardBody([
                    dbc.Row([
                        dbc.Col([
                            dcc.Dropdown(
                                id='store-dropdown',
                                options=[{'label': f'Store {i}', 'value': i}
                                         for i in sorted(df_open['Store'].unique())],
                                value=1,
                                clearable=False,
                                style={'marginBottom': '10px'}
                            )
                        ], width=3),
                        dbc.Col([
                            html.P("Select any store to see actual vs predicted sales",
                                   style={'color': '#666', 'marginTop': '8px'})
                        ], width=9)
                    ]),
                    dcc.Graph(id='forecast-chart')
                ])
            ])
        ], width=12)
    ]),

    html.Br()

], fluid=True, style={'backgroundColor': COLORS['light']})

# ── Callback — Forecast Chart ─────────────────────────────────────
@app.callback(
    Output('forecast-chart', 'figure'),
    Input('store-dropdown', 'value')
)
def update_forecast(store_id):
    # Get test data for selected store
    df_features['Date'] = pd.to_datetime(
        df_features[['Year', 'Month', 'Day']])
    
    store_data = df_features[
        (df_features['Store'] == store_id) &
        (df_features['Date'] >= '2015-06-01')
    ].copy()

    if len(store_data) == 0:
        return go.Figure()

    feature_cols = [c for c in df_features.columns 
                    if c not in ['Sales', 'Date']]
    
    preds = model.predict(store_data[feature_cols],
                          num_iteration=model.best_iteration)
    preds = np.clip(preds, 0, None)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=store_data['Date'], y=store_data['Sales'],
        name='Actual', line=dict(color=COLORS['primary'], width=2),
        mode='lines+markers', marker=dict(size=4)
    ))
    fig.add_trace(go.Scatter(
        x=store_data['Date'], y=preds,
        name='Predicted', 
        line=dict(color=COLORS['danger'], width=2, dash='dash'),
        mode='lines+markers', marker=dict(size=4)
    ))
    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(orientation='h'),
        xaxis_title='Date',
        yaxis_title='Sales (€)',
        margin=dict(t=10)
    )
    return fig

print("✅ Dashboard layout built successfully!")

✅ Dashboard layout built successfully!


## Step 5 — Run Dashboard

In [5]:
# ── Run dashboard ─────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8050)